# 🐦 Twitter Sentiment Analysis using NLP

**A complete end-to-end machine learning pipeline for classifying tweet sentiment.**

| | |
|---|---|
| **Task** | Multi-class text classification (Positive / Negative / Neutral) |
| **Models** | Logistic Regression, Multinomial Naive Bayes |
| **Features** | TF-IDF (unigrams + bigrams) |
| **Preprocessing** | NLTK — tokenise, stopwords, lemmatise |

---
## Table of Contents
1. [Setup & Imports](#1)
2. [Load & Explore Data](#2)
3. [Text Preprocessing](#3)
4. [Exploratory Data Analysis (EDA)](#4)
5. [Feature Engineering — TF-IDF](#5)
6. [Model Training](#6)
7. [Evaluation & Comparison](#7)
8. [Inference — Predict New Tweets](#8)
9. [Save Model](#9)

## 1. Setup & Imports <a id='1'></a>

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# Project modules
PROJECT_ROOT = os.path.dirname(os.getcwd())   # notebooks/ → project root
sys.path.insert(0, PROJECT_ROOT)
from src.preprocessing import TextPreprocessor
from src.utils import load_data, encode_labels, save_artifact, SENTIMENT_COLORS

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All imports successful!')

## 2. Load & Explore Data <a id='2'></a>

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'tweets.csv')
df = load_data(DATA_PATH)

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
df.head(8)

In [ ]:
# Class distribution
print('Class distribution:')
print(df['sentiment'].value_counts())
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Tweet length statistics
df['tweet_length'] = df['text'].str.len()
df['word_count']   = df['text'].str.split().str.len()

df.groupby('sentiment')[['tweet_length', 'word_count']].describe().round(1)

## 3. Text Preprocessing <a id='3'></a>

In [ ]:
# Demonstrate preprocessing steps
sample = "I LOVE this new phone! 😍 https://example.com #tech @Apple It's amazing!"
print(f'Raw text  : {sample}')

preprocessor = TextPreprocessor(remove_stops=True, lemmatize=True)
cleaned = preprocessor.clean(sample)
print(f'Cleaned   : {cleaned}')

In [ ]:
# Apply preprocessing to the entire dataset
print('Preprocessing all tweets...')
df['cleaned_text'] = preprocessor.clean_series(df['text'])

# Remove rows where cleaning produced empty strings
df = df[df['cleaned_text'].str.strip() != ''].reset_index(drop=True)
print(f'Rows after cleaning: {len(df):,}')
df[['text', 'cleaned_text', 'sentiment']].head(6)

## 4. Exploratory Data Analysis (EDA) <a id='4'></a>

In [ ]:
# ── Sentiment distribution bar chart ──────────────────────────────────────
counts = df['sentiment'].value_counts()
colors = [SENTIMENT_COLORS.get(s, '#95a5a6') for s in counts.index]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.index, counts.values, color=colors,
               edgecolor='white', linewidth=1.5, width=0.55)

total = counts.sum()
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + total*0.005,
            f'{val}\n({val/total*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Class');  ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout();  plt.show()

In [ ]:
# ── Tweet length distribution by class ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title in zip(axes,
                           ['tweet_length', 'word_count'],
                           ['Character Count', 'Word Count']):
    for sentiment, color in SENTIMENT_COLORS.items():
        subset = df[df['sentiment'] == sentiment][col]
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=sentiment)
    ax.set_title(f'{title} by Sentiment', fontsize=12)
    ax.set_xlabel(title); ax.set_ylabel('Frequency')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout();  plt.show()

In [ ]:
# ── Word clouds ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cmap_map = {'positive': 'Greens', 'negative': 'Reds', 'neutral': 'Blues'}

for ax, sentiment in zip(axes, ['positive', 'negative', 'neutral']):
    text = ' '.join(df[df['sentiment'] == sentiment]['cleaned_text'])
    wc = WordCloud(
        width=600, height=350,
        background_color='white',
        colormap=cmap_map[sentiment],
        max_words=100, collocations=False
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sentiment.capitalize()} Tweets',
                 fontsize=13, fontweight='bold',
                 color=SENTIMENT_COLORS[sentiment])

plt.suptitle('Word Clouds by Sentiment Class', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout();  plt.show()

## 5. Feature Engineering — TF-IDF <a id='5'></a>

In [ ]:
# Encode labels
y, label_map = encode_labels(df['sentiment'])
label_names  = [k for k, _ in sorted(label_map.items(), key=lambda x: x[1])]
print(f'Label map : {label_map}')
print(f'Labels    : {label_names}')

# Train / test split (stratified)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_text'], y,
    test_size=0.20, random_state=42, stratify=y
)
print(f'\nTrain size : {len(X_train_raw):,}')
print(f'Test size  : {len(X_test_raw):,}')

In [ ]:
# Build and fit TF-IDF vectoriser
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),    # unigrams + bigrams
    sublinear_tf=True,     # log normalisation
    min_df=2,
    strip_accents='unicode'
)

X_train = vectorizer.fit_transform(X_train_raw)
X_test  = vectorizer.transform(X_test_raw)

print(f'Vocabulary size  : {len(vectorizer.vocabulary_):,}')
print(f'X_train shape    : {X_train.shape}')
print(f'X_test  shape    : {X_test.shape}')

# Top TF-IDF features
feature_names = vectorizer.get_feature_names_out()
print(f'\nSample features  : {feature_names[:20]}')

## 6. Model Training <a id='6'></a>

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0, solver='lbfgs', random_state=42
    ),
    'Naive Bayes': MultinomialNB(alpha=0.5),
}

results     = {}
predictions = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    results[name]     = acc
    predictions[name] = y_pred

    # Cross-validation
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f'  Test accuracy : {acc:.4f}')
    print(f'  5-fold CV     : {cv.mean():.4f} ± {cv.std():.4f}\n')

## 7. Evaluation & Comparison <a id='7'></a>

In [ ]:
# ── Full classification reports ────────────────────────────────────────────
for name, y_pred in predictions.items():
    print(f'\n{'─'*55}')
    print(f'  {name}')
    print(f'{'─'*55}')
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

In [ ]:
# ── Confusion matrices side by side ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(f'Confusion Matrix\n{name}', fontsize=12, fontweight='bold')

plt.tight_layout();  plt.show()

In [ ]:
# ── Model accuracy comparison ─────────────────────────────────────────────
import matplotlib.patches as mpatches

model_names = list(results.keys())
accuracies  = [results[m] for m in model_names]
max_acc     = max(accuracies)
bar_colors  = ['#2ecc71' if a == max_acc else '#3498db' for a in accuracies]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(model_names, [a*100 for a in accuracies],
               color=bar_colors, edgecolor='white', height=0.45)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_width()-0.8, bar.get_y()+bar.get_height()/2,
            f'{acc*100:.2f}%', va='center', ha='right',
            color='white', fontsize=11, fontweight='bold')

ax.set_xlim(0, 105)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)

best_patch = mpatches.Patch(color='#2ecc71', label='Best Model')
ax.legend(handles=[best_patch], loc='lower right')
plt.tight_layout();  plt.show()

best_name = max(results, key=results.get)
print(f'\n🏆 Best model: {best_name}  ({results[best_name]*100:.2f}%)')

## 8. Inference — Predict New Tweets <a id='8'></a>

In [ ]:
best_model = models[best_name]
reverse_map = {v: k for k, v in label_map.items()}

def predict_tweet(text: str) -> dict:
    """Quick inline predict for notebook experimentation."""
    cleaned  = preprocessor.clean(text)
    features = vectorizer.transform([cleaned])
    proba    = best_model.predict_proba(features)[0]
    idx      = proba.argmax()
    return {
        'text'      : text,
        'sentiment' : reverse_map[idx],
        'confidence': round(float(proba.max()), 4),
        'probs'     : {reverse_map[i]: round(float(p),4) for i,p in enumerate(proba)}
    }

# Test a few examples
test_tweets = [
    'I absolutely love this product! Works perfectly every time 🎉',
    'Worst customer service ever. Waited 2 hours, zero help!',
    'My order arrived on Tuesday.',
    'Cannot believe how amazing this new feature is!!!',
    'The app keeps crashing and nobody responds to support tickets.',
]

print(f'Using best model: {best_name}\n')
for tweet in test_tweets:
    r = predict_tweet(tweet)
    print(f"Tweet      : {r['text']}")
    print(f"Sentiment  : {r['sentiment'].upper()}  (confidence: {r['confidence']*100:.1f}%)")
    print(f"Probs      : {r['probs']}")
    print()

## 9. Save Model <a id='9'></a>

In [ ]:
MODEL_PATH = os.path.join(PROJECT_ROOT, 'model', 'model.pkl')

model_bundle = {
    'vectorizer'      : vectorizer,
    'model'           : best_model,
    'label_map'       : label_map,
    'label_names'     : label_names,
    'preprocessor'    : preprocessor,
    'best_model_name' : best_name,
    'test_accuracy'   : results[best_name],
}

save_artifact(model_bundle, MODEL_PATH)
print(f'✅ Model bundle saved to: {MODEL_PATH}')
print(f'   Best model : {best_name}')
print(f'   Accuracy   : {results[best_name]*100:.2f}%')

---
## ✅ Summary

| Step | Details |
|------|----------|
| Dataset | Custom Twitter sentiment CSV |
| Preprocessing | Lowercase → strip URLs/mentions → remove punct/nums → tokenise → stopwords → lemmatise |
| Features | TF-IDF (unigrams + bigrams, 10 000 features) |
| Models | Logistic Regression, Multinomial Naive Bayes |
| Best model | Selected automatically by test accuracy |
| Saved artifacts | `model/model.pkl` (vectoriser + model + label map + preprocessor) |

Run the Streamlit app:
```bash
streamlit run app.py
```